# FSR chunk-level ESN / equipment mismatch — statistics

**Trigger:** 2026-05-27 Jon/Tao/Vince meeting prep. Sample row showed:

- Top-level `esn = "295456"` (Gas Turbine)
- Same chunk's `metadata.esn = "336X233"` (Generator)
- `chunk_text` is generator stator winding HIPOT content

PDF: `Field_Service_Report_ProjectID_EV-114550_EVP-541831.pdf`. The screenshot called it *"one of 11,300 rows for our PDF"*.

This notebook quantifies, across the whole chunks table:

1. How often does top-level `esn` disagree with `metadata.esn`?
2. How many distinct `metadata.esn` values appear per PDF?
3. How often does top-level `equipment_type` disagree with `metadata.equipment_type` (the Gen-content-filed-under-GT problem at chunk level)?
4. Same drill, scoped to the specific PDF from the screenshot.

**Tables (prod):**
- Metadata: `vaip.ai_sot_field_service_report.biz_metadata_field_service_report`
- Chunks:   `vaip.ai_std_con_field_service_report.vec_field_service_report`

Schema reference: [pw_sdg_ai_ser_repo/common/fsr_config.py](../../../pw_sdg_ai_ser_repo/common/fsr_config.py) — `CHUNK_TABLE_DDL_COLS`. Top-level columns: `chunk_id`, `chunk_index`, `document_id`, `pdf_name`, `page_number`, `chunk_text`, `esn`, `report_date`, `chunk_embedding`, `metadata` (STRING — JSON blob).

Run on Databricks SQL warehouse or a serverless cluster — pure SQL, no Python.

## 0. Config — point at the right env

Default = prod (`vaip`). For dev runs, swap catalog to `vaid` and refresh `TARGET_PDF` from a dev document.

In [ ]:
CATALOG        = "vaid"   # vaid / vaiq / vais / vaip
METADATA_TABLE = f"{CATALOG}.ai_sot_field_service_report.biz_metadata_field_service_report"
CHUNK_TABLE    = f"{CATALOG}.ai_std_con_field_service_report.vec_field_service_report"

# PDF from the screenshot — used for the per-PDF drill in §6.
TARGET_PDF     = "Field_Service_Report_ProjectID_EV-114550_EVP-541831.pdf"

print(f"Metadata: {METADATA_TABLE}")
print(f"Chunks  : {CHUNK_TABLE}")
print(f"Target  : {TARGET_PDF}")

## 1. Sanity — table size + schema

Confirms the table is reachable and `metadata` is the JSON STRING column we expect.

In [ ]:
display(spark.sql(f"DESCRIBE {CHUNK_TABLE}"))

In [ ]:
display(spark.sql(f"""
SELECT
  COUNT(*)                                AS total_chunks,
  COUNT(DISTINCT document_id)             AS distinct_docs,
  COUNT(DISTINCT pdf_name)                AS distinct_pdfs,
  COUNT(DISTINCT esn)                     AS distinct_top_level_esns,
  SUM(CASE WHEN esn IS NULL OR TRIM(esn) = '' THEN 1 ELSE 0 END) AS chunks_with_null_esn,
  SUM(CASE WHEN metadata IS NULL THEN 1 ELSE 0 END)              AS chunks_with_null_metadata
FROM {CHUNK_TABLE}
"""))

## 2. Parse `metadata` JSON once into a working view

`metadata` is a STRING column holding a JSON object. The fields we care about (per the screenshot): `esn`, `equipment_sys_id`, `equipment_type`, `equipment_class_code`. Use `get_json_object` so this is portable across runtimes.

Persists as a temp view `chunks_parsed` for the rest of the notebook.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW chunks_parsed AS
SELECT
  chunk_id,
  document_id,
  pdf_name,
  page_number,
  chunk_index,
  esn                                              AS esn_top,
  get_json_object(metadata, '$.esn')               AS esn_meta,
  get_json_object(metadata, '$.equipment_type')    AS eqtype_meta,
  get_json_object(metadata, '$.equipment_sys_id')  AS eqsysid_meta,
  get_json_object(metadata, '$.equipment_class_code') AS eqclass_meta,
  chunk_text
FROM {CHUNK_TABLE}
""")
print("View ready: chunks_parsed")
display(spark.sql("SELECT * FROM chunks_parsed LIMIT 5"))

## 3. Headline mismatch — top-level `esn` vs `metadata.esn`

If the pipeline propagated the same ESN everywhere, mismatch rate ≈ 0. Anything material here = empirical evidence that one PDF carries multiple ESNs and chunks know which one they belong to better than the top-level column does.

Counts are normalized (trim + lower) to avoid case/whitespace noise.

In [ ]:
display(spark.sql("""
WITH n AS (
  SELECT
    LOWER(TRIM(esn_top))  AS esn_top_n,
    LOWER(TRIM(esn_meta)) AS esn_meta_n
  FROM chunks_parsed
)
SELECT
  COUNT(*)                                                                       AS total_chunks,
  SUM(CASE WHEN esn_top_n IS NULL  OR esn_top_n  = '' THEN 1 ELSE 0 END)         AS top_null_or_blank,
  SUM(CASE WHEN esn_meta_n IS NULL OR esn_meta_n = '' THEN 1 ELSE 0 END)         AS meta_null_or_blank,
  SUM(CASE WHEN esn_top_n IS NOT NULL AND esn_meta_n IS NOT NULL
            AND esn_top_n = esn_meta_n THEN 1 ELSE 0 END)                        AS match_count,
  SUM(CASE WHEN esn_top_n IS NOT NULL AND esn_meta_n IS NOT NULL
            AND esn_top_n <> esn_meta_n THEN 1 ELSE 0 END)                       AS mismatch_count,
  ROUND( 100.0 * SUM(CASE WHEN esn_top_n IS NOT NULL AND esn_meta_n IS NOT NULL
                        AND esn_top_n <> esn_meta_n THEN 1 ELSE 0 END) / COUNT(*), 2)
                                                                                  AS pct_mismatch
FROM n
"""))

## 4. Multi-ESN per PDF — distinct `metadata.esn` per `pdf_name`

How many PDFs carry >1 distinct ESN at chunk level? This is the empirical multi-ESN cardinality (different from §3 which compares top-level vs metadata).

Buckets: 1 / 2 / 3 / 4 / 5+ distinct ESNs.

In [ ]:
display(spark.sql("""
WITH per_pdf AS (
  SELECT
    pdf_name,
    COUNT(DISTINCT LOWER(TRIM(esn_meta))) AS n_distinct_esns
  FROM chunks_parsed
  WHERE esn_meta IS NOT NULL AND TRIM(esn_meta) <> ''
  GROUP BY pdf_name
)
SELECT
  CASE
    WHEN n_distinct_esns = 1 THEN '1 ESN'
    WHEN n_distinct_esns = 2 THEN '2 ESNs'
    WHEN n_distinct_esns = 3 THEN '3 ESNs'
    WHEN n_distinct_esns = 4 THEN '4 ESNs'
    ELSE '5+ ESNs'
  END                       AS bucket,
  COUNT(*)                  AS n_pdfs,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM per_pdf
GROUP BY 1
ORDER BY MIN(n_distinct_esns)
"""))

## 5. Equipment-type drift — top-level vs metadata

The big retrieval-quality question: how often is a chunk **tagged with a different equipment type at the top level than what its own metadata says it is**? This is the chunk-level version of the Gen-content-filed-under-GT problem (FSR/ESN-resolution/train investigation, 458 PDFs).

Top-level `equipment_type` is not on the chunk row — it lives on metadata registry — so we join through `document_id`. Compare against `metadata.equipment_type` per chunk.

In [ ]:
display(spark.sql(f"""
WITH doc_eq AS (
  SELECT document_id, equipment_type AS eqtype_doc
  FROM {METADATA_TABLE}
)
SELECT
  d.eqtype_doc                              AS doc_level_equipment_type,
  c.eqtype_meta                             AS chunk_metadata_equipment_type,
  COUNT(*)                                  AS chunk_count
FROM chunks_parsed c
LEFT JOIN doc_eq d USING (document_id)
GROUP BY 1, 2
ORDER BY chunk_count DESC
"""))

**Headline:** how many chunks have a Generator-classed metadata but live in a Gas-Turbine-tagged document (and vice versa)?

In [ ]:
display(spark.sql(f"""
WITH doc_eq AS (
  SELECT document_id, equipment_type AS eqtype_doc
  FROM {METADATA_TABLE}
)
SELECT
  COUNT(*) AS total_chunks_with_both_known,
  SUM(CASE WHEN d.eqtype_doc <> c.eqtype_meta THEN 1 ELSE 0 END)                       AS chunks_eq_mismatch,
  SUM(CASE WHEN d.eqtype_doc ILIKE '%turbine%' AND c.eqtype_meta ILIKE '%generator%' THEN 1 ELSE 0 END) AS gen_chunk_in_gt_doc,
  SUM(CASE WHEN d.eqtype_doc ILIKE '%generator%' AND c.eqtype_meta ILIKE '%turbine%' THEN 1 ELSE 0 END) AS gt_chunk_in_gen_doc,
  ROUND(100.0 * SUM(CASE WHEN d.eqtype_doc <> c.eqtype_meta THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mismatch
FROM chunks_parsed c
JOIN doc_eq d USING (document_id)
WHERE d.eqtype_doc IS NOT NULL AND c.eqtype_meta IS NOT NULL
"""))

## 6. Drill — the specific PDF from the screenshot

`Field_Service_Report_ProjectID_EV-114550_EVP-541831.pdf`. Confirms what's actually in the chunks for this one PDF and surfaces the per-page ESN/equipment-type story.

In [ ]:
display(spark.sql(f"""
SELECT
  COUNT(*)                                            AS n_chunks,
  COUNT(DISTINCT page_number)                         AS n_pages,
  COUNT(DISTINCT LOWER(TRIM(esn_top)))                AS n_distinct_esn_top,
  COUNT(DISTINCT LOWER(TRIM(esn_meta)))               AS n_distinct_esn_meta,
  COUNT(DISTINCT eqtype_meta)                         AS n_distinct_eqtype_meta
FROM chunks_parsed
WHERE pdf_name = '{TARGET_PDF}'
"""))

In [ ]:
display(spark.sql(f"""
SELECT
  esn_top,
  esn_meta,
  eqtype_meta,
  COUNT(*)             AS chunk_count,
  MIN(page_number)     AS first_page,
  MAX(page_number)     AS last_page
FROM chunks_parsed
WHERE pdf_name = '{TARGET_PDF}'
GROUP BY esn_top, esn_meta, eqtype_meta
ORDER BY chunk_count DESC
"""))

## 7. Spot-check sample — mismatched rows across the corpus

15 random chunks where top-level `esn` ≠ `metadata.esn`. Useful to paste into the meeting if Jon wants live examples beyond the screenshot.

In [ ]:
display(spark.sql("""
SELECT
  pdf_name,
  page_number,
  esn_top,
  esn_meta,
  eqtype_meta,
  SUBSTR(chunk_text, 1, 200) AS chunk_text_preview
FROM chunks_parsed
WHERE esn_top  IS NOT NULL AND TRIM(esn_top)  <> ''
  AND esn_meta IS NOT NULL AND TRIM(esn_meta) <> ''
  AND LOWER(TRIM(esn_top)) <> LOWER(TRIM(esn_meta))
ORDER BY RAND()
LIMIT 15
"""))

## 8. Headline numbers — copy/paste block for the meeting

Run after §3, §4, §5b have populated. Pulls the four headline metrics into a single result.

In [ ]:
display(spark.sql(f"""
WITH
  c AS (
    SELECT
      LOWER(TRIM(esn_top))  AS esn_top_n,
      LOWER(TRIM(esn_meta)) AS esn_meta_n,
      document_id, pdf_name, eqtype_meta
    FROM chunks_parsed
  ),
  doc_eq AS (
    SELECT document_id, equipment_type AS eqtype_doc FROM {METADATA_TABLE}
  ),
  esn_mm AS (
    SELECT
      COUNT(*)                                                                        AS total_chunks,
      SUM(CASE WHEN esn_top_n IS NOT NULL AND esn_meta_n IS NOT NULL
                AND esn_top_n <> esn_meta_n THEN 1 ELSE 0 END)                        AS esn_mismatch_chunks
    FROM c
  ),
  pdf_multi AS (
    SELECT COUNT(*) AS multi_esn_pdfs
    FROM (
      SELECT pdf_name
      FROM c WHERE esn_meta_n IS NOT NULL AND esn_meta_n <> ''
      GROUP BY pdf_name HAVING COUNT(DISTINCT esn_meta_n) > 1
    )
  ),
  eq_mm AS (
    SELECT
      SUM(CASE WHEN d.eqtype_doc <> c.eqtype_meta THEN 1 ELSE 0 END)                  AS eqtype_mismatch_chunks,
      SUM(CASE WHEN d.eqtype_doc ILIKE '%turbine%' AND c.eqtype_meta ILIKE '%generator%' THEN 1 ELSE 0 END) AS gen_chunks_in_gt_docs
    FROM c JOIN doc_eq d USING (document_id)
    WHERE d.eqtype_doc IS NOT NULL AND c.eqtype_meta IS NOT NULL
  )
SELECT
  esn_mm.total_chunks,
  esn_mm.esn_mismatch_chunks,
  ROUND(100.0 * esn_mm.esn_mismatch_chunks / esn_mm.total_chunks, 2)                 AS pct_esn_mismatch,
  pdf_multi.multi_esn_pdfs,
  eq_mm.eqtype_mismatch_chunks,
  eq_mm.gen_chunks_in_gt_docs
FROM esn_mm, pdf_multi, eq_mm
"""))